# Traffic Signs Recognition using CNN and Keras

Trains a convolutional neural network from scratch to classify German traffic
signs (GTSRB dataset — 43 classes). Built to run on Google Colab's free GPU.

**Steps in this notebook:**
1. Set up Kaggle API + download the GTSRB dataset
2. Load and preprocess images
3. Build a CNN from scratch (no transfer learning)
4. Train with train/validation split
5. Evaluate on the official test set + confusion matrix
6. Save the trained model


## 1. Setup — install deps and connect Kaggle API

Upload your `kaggle.json` API token when prompted (Kaggle account -> Settings -> Create New API Token).

In [ ]:
!pip install -q kaggle

from google.colab import files
uploaded = files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


## 2. Download and extract the GTSRB dataset

In [ ]:
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -q gtsrb-german-traffic-sign.zip -d data
!ls data


## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

print("GPU available:", tf.config.list_physical_devices('GPU'))


## 4. Load and preprocess images

Resizes every image to 32x32 and normalizes pixel values to [0, 1].
GTSRB has 43 classes, one folder per class under `data/Train`.

In [ ]:
IMG_SIZE = 32
NUM_CLASSES = 43
DATA_DIR = "data/Train"

images = []
labels = []

for class_id in range(NUM_CLASSES):
    class_dir = os.path.join(DATA_DIR, str(class_id))
    if not os.path.isdir(class_dir):
        continue
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(class_id)

images = np.array(images, dtype="float32") / 255.0
labels = np.array(labels)

print("Total images:", images.shape[0])
print("Image shape:", images.shape[1:])


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)

print("Train:", X_train.shape, "Val:", X_val.shape)


## 5. Build the CNN

A compact architecture from scratch — no transfer learning, since the goal here is to show the model can be designed and trained end-to-end.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation="relu"),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 6. Train

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()


## 7. Evaluate on the official GTSRB test set

In [ ]:
test_df = pd.read_csv("data/Test.csv")

test_images = []
test_labels = []

for _, row in test_df.iterrows():
    img_path = os.path.join("data", row["Path"])
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    test_images.append(img)
    test_labels.append(row["ClassId"])

X_test = np.array(test_images, dtype="float32") / 255.0
y_test = np.array(test_labels)

test_loss, test_acc = model.evaluate(X_test, to_categorical(y_test, NUM_CLASSES))
print(f"Test accuracy: {test_acc:.4f}")


In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 10))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()


## 8. Save the trained model

Download `traffic_sign_model.keras` from the Colab file browser afterward, and place it in the repo's `models/` folder for use with `src/predict.py`.

In [ ]:
model.save("traffic_sign_model.keras")
print("Saved traffic_sign_model.keras")
